Nexar Collision Prediction — Training

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


class NexarDataset(Dataset):
    """
    Loads preprocessed .npy files (frames, segmentation, depth) for each video
    and assembles them into a [T, 5, H, W] tensor.
    """

    def __init__(
        self,
        csv_path,
        frames_dir,
        seg_dir,
        depth_dir,
        split="train",
        fps=10,
        clip_len=100,
    ):
        self.frames_dir = frames_dir
        self.seg_dir = seg_dir
        self.depth_dir = depth_dir
        self.split = split
        self.fps = fps
        self.clip_len = clip_len

        df = pd.read_csv(csv_path)
        df["id"] = df["id"].astype(str).str.zfill(5)
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_id = row["id"]

        frames = np.load(os.path.join(self.frames_dir, f"{video_id}.npy"))
        seg = np.load(os.path.join(self.seg_dir, f"{video_id}.npy"))
        depth = np.load(os.path.join(self.depth_dir, f"{video_id}.npy"))

        N = len(frames)
        if self.split == "train":
            toe = row.get("time_of_event")
            if pd.notna(toe):
                end = min(int(toe * self.fps), N)
                start = max(end - self.clip_len, 0)
            else:
                start = max(N - self.clip_len, 0)
                end = N
        else:
            start, end = 0, N

        frames = frames[start:end]
        seg = seg[start:end]
        depth = depth[start:end]

        T = len(frames)
        if T < self.clip_len:
            pad = self.clip_len - T
            frames = np.concatenate(
                [np.zeros((pad, *frames.shape[1:]), dtype=frames.dtype), frames], axis=0
            )
            seg = np.concatenate(
                [np.zeros((pad, *seg.shape[1:]), dtype=seg.dtype), seg], axis=0
            )
            depth = np.concatenate(
                [np.zeros((pad, *depth.shape[1:]), dtype=depth.dtype), depth], axis=0
            )

        frames_t = torch.from_numpy(frames).permute(0, 3, 1, 2).float() / 255.0
        seg_t = torch.from_numpy(seg.astype(np.float32)).unsqueeze(1)
        depth_t = torch.from_numpy(depth.astype(np.float32)).unsqueeze(1)

        video = torch.cat([frames_t, depth_t, seg_t], dim=1)  # [T, 5, H, W]

        if self.split == "test":
            return video, video_id

        label = torch.tensor(row["target"], dtype=torch.float32)
        return video, label


print("Inline NexarDataset defined successfully.")

In [ ]:
import os

BASE = "/hpc/group/coursess26/ids705/team-project/detect-to-protect"

PATHS = {
    "train_csv": f"{BASE}/data/train.csv",
    "test_csv": f"{BASE}/data/test.csv",
    "frames_train": f"{BASE}/data/frames/train",
    "frames_test": f"{BASE}/data/frames/test",
    "seg_train": f"{BASE}/data/segmentation/train",
    "seg_test": f"{BASE}/data/segmentation/test",
    "depth_train": f"{BASE}/data/depth/train",
    "depth_test": f"{BASE}/data/depth/test",
    "output_dir": f"{BASE}/outputs",
}

os.makedirs(PATHS["output_dir"], exist_ok=True)

for k, v in PATHS.items():
    exists = os.path.exists(v)
    print(f"[{'OK' if exists else 'MISSING'}] {k:12s} -> {v}")

In [ ]:
# Basic path checks
required = [
    PATHS["train_csv"],
    PATHS["frames_train"],
    PATHS["seg_train"],
    PATHS["depth_train"],
]

for p in required:
    exists = os.path.exists(p)
    print(f'[{"OK" if exists else "MISSING"}] {p}')

assert all(
    os.path.exists(p) for p in required
), "One or more required train paths are missing."

**Sanity check - dataset**

In [ ]:
# Create dataset and inspect one sample
train_ds = NexarDataset(
    csv_path=PATHS["train_csv"],
    frames_dir=PATHS["frames_train"],
    seg_dir=PATHS["seg_train"],
    depth_dir=PATHS["depth_train"],
    split="train",
    fps=10,
    clip_len=100,
)

video, label = train_ds[0]
print(f"shape  : {video.shape}")  # [100, 5, 180, 320]
print(f"dtype  : {video.dtype}")
print(f"label  : {label.item()}")
print(f"frames : {video[:, :3].min():.3f} - {video[:, :3].max():.3f}")
print(f"depth  : {video[:, 3].min():.3f} - {video[:, 3].max():.3f}")
print(f"seg    : {video[:, 4].unique().tolist()}")

In [ ]:
# DataLoader batch check
loader = DataLoader(
    train_ds,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
batch_video, batch_label = next(iter(loader))
print(f"batch shape : {batch_video.shape}")  # [4, 100, 5, 180, 320]
print(f"labels      : {batch_label.tolist()}")

**Training**

**Train/validation split and loaders**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from torch.utils.data import Subset

SEED = 42
BATCH_SIZE = 4
NUM_WORKERS = 0  # Drive-mounted .npy files are safer with single-process loading
PIN_MEMORY = torch.cuda.is_available()

# Demo-mode dataset caps to keep end-to-end runtime short.
MAX_TRAIN_VIDEOS = 80
MAX_VAL_VIDEOS = 40

labels_np = train_ds.df["target"].astype(int).values
all_idx = np.arange(len(train_ds))

train_idx, val_idx = train_test_split(
    all_idx,
    test_size=0.2,
    random_state=SEED,
    stratify=labels_np,
)

rng = np.random.default_rng(SEED)
if len(train_idx) > MAX_TRAIN_VIDEOS:
    train_idx = rng.choice(train_idx, size=MAX_TRAIN_VIDEOS, replace=False)
if len(val_idx) > MAX_VAL_VIDEOS:
    val_idx = rng.choice(val_idx, size=MAX_VAL_VIDEOS, replace=False)

train_subset = Subset(train_ds, train_idx)
val_subset = Subset(train_ds, val_idx)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print(f"train samples: {len(train_subset)}")
print(f"val samples  : {len(val_subset)}")
print(f"pin_memory   : {PIN_MEMORY}")
print(f"demo caps    : train={MAX_TRAIN_VIDEOS}, val={MAX_VAL_VIDEOS}")

**Lightweight video-transformer baseline**

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F


class TinyVideoTransformer(nn.Module):
    def __init__(
        self, in_channels=5, embed_dim=128, num_heads=4, num_layers=2, frame_stride=5
    ):
        super().__init__()
        self.frame_stride = frame_stride

        # Per-frame spatial encoder -> D-dimensional token
        self.spatial = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, embed_dim, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + 256, embed_dim))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(embed_dim, 1)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        # x: [B, T, 5, H, W]
        x = x[:, :: self.frame_stride]
        b, t, c, h, w = x.shape

        x = x.reshape(b * t, c, h, w)
        x = self.spatial(x).flatten(1)
        x = x.reshape(b, t, -1)

        cls = self.cls_token.expand(b, -1, -1)
        x = torch.cat([cls, x], dim=1)

        if x.size(1) > self.pos_embed.size(1):
            raise ValueError("Sequence too long for positional embedding.")
        x = x + self.pos_embed[:, : x.size(1)]

        x = self.transformer(x)
        cls_out = x[:, 0]
        logits = self.head(cls_out).squeeze(1)
        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TinyVideoTransformer().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"device      : {device}")
print(f"parameters  : {n_params:,}")

In [ ]:
from tqdm.auto import tqdm

EPOCHS = 3
best_val_auc = 0.0
best_path = f"{PATHS['output_dir']}/best_tiny_video_transformer.pt"

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for batch_video, batch_label in tqdm(
        train_loader, desc=f"Epoch {epoch}/{EPOCHS} - train", leave=False
    ):
        batch_video = batch_video.to(device, non_blocking=True)
        batch_label = batch_label.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(batch_video)
            loss = criterion(logits, batch_label)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * batch_video.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    y_true, y_score = [], []

    with torch.no_grad():
        for batch_video, batch_label in tqdm(
            val_loader, desc=f"Epoch {epoch}/{EPOCHS} - val", leave=False
        ):
            batch_video = batch_video.to(device, non_blocking=True)
            batch_label = batch_label.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                logits = model(batch_video)
                loss = criterion(logits, batch_label)

            probs = torch.sigmoid(logits)
            val_loss += loss.item() * batch_video.size(0)
            y_true.extend(batch_label.detach().cpu().numpy().tolist())
            y_score.extend(probs.detach().cpu().numpy().tolist())

    val_loss /= len(val_loader.dataset)
    val_auc = roc_auc_score(y_true, y_score)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_auc": val_auc,
            },
            best_path,
        )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_auc={val_auc:.4f} | "
        f"best_val_auc={best_val_auc:.4f}"
    )

print(f"Saved best checkpoint to: {best_path}")

In [ ]:
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print(
    f"Loaded checkpoint from epoch {ckpt['epoch']} with val_auc={ckpt['val_auc']:.4f}"
)

**Main Model**

In [ ]:
# VideoMAE quick e2e (simplified)

import importlib.util
import os
import subprocess
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

if importlib.util.find_spec("transformers") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40.0"]
    )

from transformers import VideoMAEForVideoClassification

# 1) Load model
MODEL_CANDIDATES = [
    {"id": "MCG-NJU/videomae-base-finetuned-kinetics", "revision": "main"}
]
model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_CANDIDATES[0]["id"],
    revision=MODEL_CANDIDATES[0]["revision"],
    num_labels=1,
    ignore_mismatched_sizes=True,
)

# Train classifier head only
for n, p in model.named_parameters():
    p.requires_grad = "classifier" in n

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad), lr=1e-4, weight_decay=1e-4
)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

# Quick-run settings
NUM_FRAMES = int(model.config.num_frames)
IMG_SIZE = int(model.config.image_size)
EPOCHS_VMAE = 1
MAX_TRAIN_STEPS = 25
MAX_VAL_STEPS = 10

output_dir = PATHS.get("output_dir", f"{ROOT}/outputs")
os.makedirs(output_dir, exist_ok=True)
best_path_vmae = f"{output_dir}/best_videomae_quick.pt"
best_val_auc_vmae = 0.0

mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 1, 3, 1, 1)

print(
    f"VideoMAE quick run: frames={NUM_FRAMES}, img={IMG_SIZE}, train_steps={MAX_TRAIN_STEPS}, val_steps={MAX_VAL_STEPS}"
)


def preprocess_video(batch_video: torch.Tensor) -> torch.Tensor:
    # [B, T, 5, H, W] -> [B, NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE]
    x = batch_video[:, :, :3]
    b, t, c, h, w = x.shape
    idx = torch.linspace(0, t - 1, steps=NUM_FRAMES, device=x.device).long()
    x = x.index_select(1, idx).reshape(b * NUM_FRAMES, c, h, w)
    x = F.interpolate(
        x, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False
    )
    x = x.reshape(b, NUM_FRAMES, c, IMG_SIZE, IMG_SIZE)
    return (x - mean) / std


def run_one_epoch(loader, train_mode: bool, max_steps: int):
    model.train(train_mode)
    total_loss, seen = 0.0, 0
    y_true, y_score = [], []

    with torch.set_grad_enabled(train_mode):
        for step, (batch_video, batch_label) in enumerate(loader, start=1):
            if step > max_steps:
                break

            batch_video = batch_video.to(device, non_blocking=True)
            batch_label = batch_label.to(device, non_blocking=True)
            pixel_values = preprocess_video(batch_video)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=use_amp):
                logits = model(pixel_values=pixel_values).logits.squeeze(-1)
                loss = criterion(logits, batch_label)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            probs = torch.sigmoid(logits)
            total_loss += loss.item() * batch_video.size(0)
            seen += batch_video.size(0)
            y_true.extend(batch_label.detach().cpu().tolist())
            y_score.extend(probs.detach().cpu().tolist())

    avg_loss = total_loss / max(seen, 1)
    auc = roc_auc_score(y_true, y_score) if len(set(y_true)) > 1 else 0.5
    return avg_loss, auc


for epoch in range(1, EPOCHS_VMAE + 1):
    train_loss, _ = run_one_epoch(
        tqdm(train_loader, desc=f"VideoMAE train {epoch}/{EPOCHS_VMAE}", leave=False),
        train_mode=True,
        max_steps=MAX_TRAIN_STEPS,
    )
    val_loss, val_auc = run_one_epoch(
        tqdm(val_loader, desc=f"VideoMAE val {epoch}/{EPOCHS_VMAE}", leave=False),
        train_mode=False,
        max_steps=MAX_VAL_STEPS,
    )

    if val_auc > best_val_auc_vmae:
        best_val_auc_vmae = val_auc
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_auc": val_auc,
                "config": {
                    "NUM_FRAMES": NUM_FRAMES,
                    "IMG_SIZE": IMG_SIZE,
                    "MAX_TRAIN_STEPS": MAX_TRAIN_STEPS,
                    "MAX_VAL_STEPS": MAX_VAL_STEPS,
                },
            },
            best_path_vmae,
        )

    print(
        f"[VideoMAE-QUICK] epoch={epoch:02d} "
        f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
        f"val_auc={val_auc:.4f} best_val_auc={best_val_auc_vmae:.4f}"
    )

print(f"Saved quick checkpoint to: {best_path_vmae}")

**Generate test predictions and submission file**

In [ ]:
import pandas as pd
from torch.utils.data import DataLoader
from transformers import VideoMAEForVideoClassification

# 1) Resolve paths and checkpoint
output_dir = PATHS.get("output_dir", f"{ROOT}/outputs")
best_path_vmae = f"{output_dir}/best_videomae_quick.pt"
submission_path = f"{output_dir}/submission_videomae_quick.csv"

# 2) Rebuild model and load trained weights
model_id = (
    MODEL_CANDIDATES[0]["id"]
    if "MODEL_CANDIDATES" in globals()
    else "MCG-NJU/videomae-base-finetuned-kinetics"
)
model = VideoMAEForVideoClassification.from_pretrained(
    model_id,
    num_labels=1,
    ignore_mismatched_sizes=True,
)

ckpt = torch.load(best_path_vmae, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(device)
model.eval()

# Use checkpoint config if available, else model defaults
cfg = ckpt.get("config", {})
NUM_FRAMES_INFER = int(cfg.get("NUM_FRAMES", model.config.num_frames))
IMG_SIZE_INFER = int(cfg.get("IMG_SIZE", model.config.image_size))

mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 1, 3, 1, 1)


def prep_for_videomae_infer(batch_video: torch.Tensor) -> torch.Tensor:
    # Input: [B, T, 5, H, W] -> Output: [B, NUM_FRAMES_INFER, 3, IMG_SIZE_INFER, IMG_SIZE_INFER]
    x = batch_video[:, :, :3]
    b, t, c, h, w = x.shape

    idx = torch.linspace(0, t - 1, steps=NUM_FRAMES_INFER, device=x.device).long()
    x = x.index_select(1, idx)

    x = x.reshape(b * NUM_FRAMES_INFER, c, h, w)
    x = torch.nn.functional.interpolate(
        x,
        size=(IMG_SIZE_INFER, IMG_SIZE_INFER),
        mode="bilinear",
        align_corners=False,
    )
    x = x.reshape(b, NUM_FRAMES_INFER, c, IMG_SIZE_INFER, IMG_SIZE_INFER)
    x = (x - mean) / std
    return x


# 3) Build test dataset/loader
test_ds = NexarDataset(
    csv_path=PATHS["test_csv"],
    frames_dir=PATHS["frames_test"],
    seg_dir=PATHS["seg_test"],
    depth_dir=PATHS["depth_test"],
    split="test",
    fps=10,
    clip_len=100,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

# 4) Predict probabilities
all_ids, all_probs = [], []
with torch.no_grad():
    for batch_video, batch_ids in tqdm(
        test_loader, desc="Inference on test", leave=False
    ):
        batch_video = batch_video.to(device, non_blocking=True)
        pixel_values = prep_for_videomae_infer(batch_video)

        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(pixel_values=pixel_values).logits.squeeze(-1)
            probs = torch.sigmoid(logits)

        all_probs.extend(probs.detach().cpu().numpy().tolist())
        all_ids.extend(list(batch_ids))

# 5) Create Kaggle submission CSV
sub_df = pd.DataFrame({"id": all_ids, "target": all_probs})
sub_df.to_csv(submission_path, index=False)

print(f"Saved submission file: {submission_path}")
print(sub_df.head())